In [1]:
import pandas as pd

In [23]:
LOBSTER_DATETIME_COLUMNS = ['datetime', 'date', 'timestamp']

def read_lobster_file(filepath: str, levels: int=1) -> pd.DataFrame:
    
    df = pd.read_csv(filepath, header=0)

    columns = ['timestamp']
    for level in range(1, levels + 1):
        columns.extend([
            f'ask_price_{level}', f'ask_size_{level}',
            f'bid_price_{level}', f'bid_size_{level}'
        ])
    df.columns = columns

    df['date'] = pd.to_datetime(filepath.split('/')[-1].split('_')[1])
    df['datetime'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['timestamp'].astype(str))

    price_cols = [col for col in columns if 'price' in col]
    for col in price_cols:
        df[col] = (pd.to_numeric(df[col], errors='coerce') / 10000.0).astype('float64') # un-scale prices

    size_cols = [col for col in columns if 'size' in col]
    for col in size_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('int64')

    df = df[LOBSTER_DATETIME_COLUMNS + columns[1:]] # reorder columns to have datetime first

    return df

In [24]:
from multiprocessing import Pool
import glob

In [25]:
def read_lobster_folder(foldername, levels: int=1) -> pd.DataFrame:

    files = glob.glob(f'{foldername}/*.csv')
    with Pool(processes=16) as pool:
        dfs = pool.map(read_lobster_file, files)

    try:
        df = pd.concat(dfs, ignore_index=True).sort_values(by='datetime').reset_index(drop=True)
    except Exception as e:
        pass

    return df

In [26]:
import os

In [32]:
def pull_lobster_data(ticker: str) -> pd.DataFrame:
    return (
        pd.read_parquet(os.path.join("processed", "AAPL", "prices.parquet"))[['datetime', 'mid_price']]
        .rename(columns={'datetime': 'date'})
        .set_index('date')
        .replace(0.0, np.nan)
    )

In [36]:
# from settings.default import ALL_QUANDL_CODES
LOBSTER_DATETIME_COLUMNS = ['datetime', 'date', 'timestamp']
LOBSTER_TICKERS = ['AAPL', 'AMZN', 'GOOG', 'INTC', 'MSFT']

import os
import pandas as pd
from multiprocessing import Pool
import glob

LEVELS = 1


def read_lobster_file(filepath: str, levels: int=1) -> pd.DataFrame:
    
    df = pd.read_csv(filepath, header=0)

    columns = ['timestamp']
    for level in range(1, levels + 1):
        columns.extend([
            f'ask_price_{level}', f'ask_size_{level}',
            f'bid_price_{level}', f'bid_size_{level}'
        ])
    df.columns = columns

    df['date'] = pd.to_datetime(filepath.split('/')[-1].split('_')[1])
    df['datetime'] = pd.to_datetime(df['date'].astype(str) + ' ' + df['timestamp'].astype(str))

    price_cols = [col for col in columns if 'price' in col]
    for col in price_cols:
        df[col] = (pd.to_numeric(df[col], errors='coerce') / 10000.0).astype('float64') # un-scale prices

    size_cols = [col for col in columns if 'size' in col]
    for col in size_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('int64')

    df = df[LOBSTER_DATETIME_COLUMNS + columns[1:]] # reorder columns to have datetime first

    return df


def read_lobster_folder(foldername, levels: int=1, workers: int=8) -> pd.DataFrame:

    files = glob.glob(f'{foldername}/*.csv')
    with Pool(processes=workers) as pool:
        dfs = pool.map(read_lobster_file, files)

    try:
        df = pd.concat(dfs, ignore_index=True).sort_values(by='datetime').reset_index(drop=True)
    except Exception as e:
        pass

    return df


def main():

    for t in LOBSTER_TICKERS:
        print(t)
        try:
            output = read_lobster_folder(f'raw/_data_dwn_43_456_transformer_{t}_2024-01-01_2025-01-01_1_1') # TODO: change folder name to match data source

            if not os.path.exists(os.path.join('processed', t)):
                os.makedirs(os.path.join('processed', t))

            output['mid_price'] = ((output['ask_price_1'] + output['bid_price_1']) / 2.0).astype('float64')

            output.to_parquet(f'processed/{t}/prices.parquet', index=False)
        except BaseException as ex:
            print(ex)


if __name__ == "__main__":
    main()

AAPL
AMZN
GOOG
INTC
MSFT
